# injection-latent-intent: does a frozen will-comply probe still fire after LoRA-DPO injection defenses?
**Branch:** `feature/injection-latent-intent` | **Gate:** pre-registered in README.md (frozen 2026-11-08)
**One-Click Execution:** Runtime -> Run all. Clones, installs, mounts Drive (results + HF cache), runs pytest on the GPU
(aborts if invalid), runs the pilot (early-abandon rule) and the resumable campaign, prints RESULTS.md, unassigns the GPU.
Re-running after a disconnect resumes from Drive. Needs an HF token in Colab Secrets (`HF_TOKEN`) with access to
meta-llama/Llama-3.1-8B-Instruct and facebook/Meta-SecAlign-8B for the secondary arm (skipped otherwise).

In [ ]:
# Step 1: Mount Drive first so the HF cache (model weights) lives there and survives reconnects
import os  # first cell: the Drive prompt appears right away
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/moonshot_results/injection-latent-intent"
os.environ["IL_RESULTS"] = BASE
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs(BASE, exist_ok=True); os.makedirs(os.environ["HF_HOME"], exist_ok=True)
# os.environ["IL_SKIP_SECONDARY"] = "1"   # uncomment to skip the Meta-SecAlign-8B arm (~70 min, not in the gate)
# os.environ["IL_FORCE"] = "1"            # uncomment to run the main campaign even if the pilot abandon rule fired
print("Results dir:", BASE, "| existing:", sorted(os.listdir(BASE)))

In [ ]:
# Step 2: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 3: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CUDA {torch.version.cuda}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/injection-latent-intent"
target_dir = "/content/injection-latent-intent"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 4: Installs (pinned transformers + peft); HF login for gated models
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# Colab ships torchao 0.10, which peft 0.21 rejects at import (needs >= 0.16); this project does not use it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print(subprocess.run([sys.executable, "-c", "import il, transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__, '| device', il.DEV, il.DTYPE)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 5: Pre-flight on the GPU (bf16): template/scorer on both tokenizers, leakage-free split, padding-invariant features,
# LoRA no-op + DPO loss = log 2 at init, end-to-end smoke, and (IL_REAL=1) real-model feature/prefill checks. ABORTS if it fails.
os.environ["IL_REAL"] = "1"
_t = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", "test_il.py"], capture_output=True, text=True)  # Colab hides child output
print(_t.stdout[-20000:]); print(_t.stderr[-5000:])
os.makedirs("/content/drive/MyDrive/moonshot_results/injection-latent-intent", exist_ok=True); open("/content/drive/MyDrive/moonshot_results/injection-latent-intent/pytest_log.txt", "w").write(_t.stdout + _t.stderr)
if _t.returncode:
    raise RuntimeError("pytest failed on GPU -> campaign aborted (INVALID)")
print("[Pre-flight] all checks passed on GPU.")

In [ ]:
# Step 6: Campaign (resumable): pilot (abandon rule) -> Qwen3-1.7B -> Llama-3.2-3B -> Meta-SecAlign-8B arm -> gate analysis
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["IL_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()